# Biber Resultate

---

- Welche Klasse erreichte durchschnittlich die meisten/wenigsten Punkte
- Welche Aufgabe wurde am besten/schlechtesten gelöst?
- Welche Punkte wurden wie häufig gegeben?
- Wie entwicklte sich die durchschnittliche Punktezahl der gesamten Schule über die Jahre?
- Wie entwicklte sich die Teilnehmeranzahl der gesamten Schule über die Jahre?

---

## LLM Setup

In [ ]:
import ollama
max_chat_history = 7
OLLAMA_API_KEY="replace-me-with-a-valid-key"
priming = """
You are the best Python coder in the world.
You know Pandas and all of Pandas functionality inside out.
You use Pandas data frames for plotting and avoid using matplotlib directly.
You code like a student in 10th grade and prefer simple solutions.
You give short and informative answers.
"""

In [ ]:
# use Ollama Cloud Models
client = ollama.Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY}
)
# OR a local Model
#client = ollama.Client(
#  host='http://127.0.0.1:11434' # Ollama und Juypter laufen in der VM
#  host='http://10.0.2.2:11434' # Ollama läuft lokal und Jupyter in der VM
#)

In [ ]:
messages = [
    {"role": "system", "content": priming}
]

from IPython.display import display, Markdown
import pandas as pd

def ask(prompt):
    """Einfacher Chat-Bot mit global spezifiziertem Client, Modell - ohne Historie"""
    global client
    global model_name
    response = client.generate(model=model_name, prompt=prompt)
    display(Markdown(response.response))

def remove_message(msg_list, n=1):
    #print("Cleanup - removing {} oldest message from chat history".format(n))
    return msg_list[0:1] + msg_list[(n+1):]
    
def chat(prompt):
    """Einfacher Chat-Bot mit global spezifiziertem Client, Modell und Historie"""
    global client
    global model_name
    global messages
    global max_chat_history
    
    messages.append({"role": "user", "content": prompt})

    if max_chat_history != 0:
        message_count = len(messages) - 1
        
        # remove exess chat history, if we have fixed size max_chat_history
        excess_messages = message_count - max_chat_history
        if max_chat_history > 0 and excess_messages > 0:
            messages = remove_message(messages, excess_messages)

        # if context window is (still) to long, we always remove old chat entries
        for _ in range(message_count + 1):
            try:
                response = client.chat(model=model_name, messages=messages)
            except ollama.ResponseError as e:
                if len(messages) > 1:
                    messages = remove_message(messages, 1)
                    continue
                raise
            except Exception as e:
                raise
    else:
        # no context lenght limit - expect your client to die eventually
        response = client.chat(model=model_name, messages=messages)
        
    messages.append({"role": "assistant", "content": response.message.content})
    display(Markdown(response.message.content))

def show_models_from_(output):
    """Gibt die Modelle von client.list() als Tabelle formatiert zurück"""
    df = pd.DataFrame([
        {
            "Model": entry.get("model", ""),
            "Size [GB]": round(int(entry.get("size", 0)) / 1024**3, 1)
        }
        for entry in output.get("models", [])
    ])

    if not df.empty:
        df = df.sort_values(by="Model", key=lambda s: s.str.lower())

    df = df.reset_index(drop=True)
    return df

In [ ]:
show_models_from_(client.list())

In [ ]:
model_name = 'gemma3:4b'

## Daten einlesen

In [ ]:
%%bash
tree

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("data/Biber-Daten-Long.csv")

In [ ]:
df

Wir entfernen unnötige Spalten

In [ ]:
df = df[['Jahr', 'Klassenstufe', 'Klasse', 'Name', 'Team', 'Aufgabe', 'Punkte']]

In [ ]:
df

In [ ]:
df["Punkte"].describe()

In [ ]:
df.groupby(['Jahr', 'Klassenstufe', 'Klasse']).describe()

## Daten analysieren

### Welche Klasse erreichte durchschnittlich die meisten/wenigsten Punkte

In [ ]:
df.pivot_table(values  = "Punkte",
               index   = ["Klasse"],
               aggfunc = ["mean"]).sort_values(by=("mean", "Punkte"))

In [ ]:
df.pivot_table(values  = "Punkte",
               index   = ["Jahr", "Klasse"],
               aggfunc = ["min", "median", "mean", "max"]).sort_values(by=("mean", "Punkte"))

### Welche Aufgabe wurde am besten/schlechtesten gelöst?

In [ ]:
chat("""
Explain this Python code:

df.pivot_table(values  = "Punkte",
               index   = ["Jahr", "Klasse"],
               aggfunc = ["min", "median", "mean", "max"]).sort_values(by=("mean", "Punkte"))
""")

### Welche Punkte wurden wie häufig gegeben?

### Wie entwickelt sich die durchschnittliche Punktezahl der Schule über die Jahre?

### Wie entwicklte sich die Teilnehmeranzahl der gesamten Schule über die Jahre?